In [24]:
# Import the dataset and turn it into dataframe from pandas
import pandas as pd
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

In [26]:
# Inspect the first 5 rows of the dataframe to learn about the dataset
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ChurnBin
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


In [27]:
# Get info about dataset non-null counts and data type of each column
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [28]:
# Change the TotalCharges column from object to numeric, and fill the empty cells with 0
# Map 0 and 1 for No and Yes from Churn column to ChurnBin
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['ChurnBin'] = df['Churn'].map({'No': 0, 'Yes': 1})

In [29]:
# Get numerical and categorical columns and define X and y
num_cols = ['tenure', 'MonthlyCharges', 'SeniorCitizen']
cat_cols = ['Contract', 'InternetService', 'PaymentMethod']
X = df[num_cols + cat_cols]
y = df['ChurnBin']

In [16]:
# split the X and y into X_train, X_test and y_train, y_test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (5282, 6), Test: (1761, 6)


In [30]:
# Build a pipeline just for numerical columns only for now
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', MinMaxScaler())
])

In [31]:
# Build a pipeline for categorical columns
from sklearn.preprocessing import OneHotEncoder

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [32]:
# Apply the two pipelines using ColumnTransformer
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

In [33]:
# Train a RandomForest model
from sklearn.ensemble import RandomForestClassifier

full_pipeline = Pipeline([
    ('processor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

full_pipeline

Pipeline(steps=[('processor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'SeniorCitizen']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Contract',
                                                   'InternetService',
                                                   'PaymentMethod'])])),
                ('model', RandomForestClassifier(random_state=42))])

In [34]:
# Fit the model
full_pipeline.fit(X_train, y_train)
print("Trained")

Trained


In [35]:
# Get the accuracy score of the model
full_pipeline.score(X_test, y_test)

0.7677455990914254

In [36]:
import joblib
joblib.dump(full_pipeline, 'rf_pipeline.joblib')
loaded = joblib.load('rf_pipeline.joblib')
print(f'Accuracy of the model: {loaded.score(X_test, y_test):.2f}')

Accuracy of the model: 0.77
